In [1]:
!pip install ragas[all]
#!pip install litellm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:

# Create a persistent models directory in Google Drive
!mkdir -p /content/drive/MyDrive/RAG/ollama_models

# Remove any existing ephemeral models directory
!rm -rf ~/.ollama/models

# Symlink Ollama's model directory to Google Drive
!ln -s /content/drive/MyDrive/RAG/ollama_models ~/.ollama/models

print('Symlinked ~/.ollama/models -> /content/drive/MyDrive/RAG/ollama_models')

import subprocess, time

server = subprocess.Popen(['ollama', 'serve'])
print('Ollama server started as background process (PID:', server.pid, ')')
time.sleep(5)
print('Waited 5 seconds for the server to initialize.')

# LLM
!ollama run gpt-oss:20b

Symlinked ~/.ollama/models -> /content/drive/MyDrive/RAG/ollama_models
Ollama server started as background process (PID: 7196 )
Waited 5 seconds for the server to initialize.
Error: 500 Internal Server Error: timed out waiting for llama runner to start - progress 0.28 - 


In [5]:
# Import libraries and setup config
import os
import pandas as pd
# from google.colab import userdata
# from openai import AsyncOpenAI
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
# Update imports to use standard metrics compatible with LangchainLLMWrapper
from ragas.metrics import ContextPrecision, ContextRecall, Faithfulness, AnswerCorrectness
# FactualCorrectness is not in ragas.metrics and requires InstructorLLM
# from ragas.metrics.collections import FactualCorrectness

# Setup LLM (Ollama)
# Ensure you have Ollama running locally or accessible.
# Example: ollama serve
# And models pulled: ollama pull llama3, ollama pull nomic-embed-text

# Initialize Langchain ChatOllama
# Replace "llama3" with your preferred model
langchain_llm = ChatOllama(model="gpt-oss:20b", temperature=0)
llm = LangchainLLMWrapper(langchain_llm)

# Initialize Langchain OllamaEmbeddings
# Replace "nomic-embed-text" with your preferred embedding model
langchain_embeddings = OllamaEmbeddings(model="nomic-embed-text")
embeddings = LangchainEmbeddingsWrapper(embeddings=langchain_embeddings)

/tmp/ipython-input-2621096982.py:11: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import ContextPrecision, ContextRecall, Faithfulness, AnswerCorrectness
/tmp/ipython-input-2621096982.py:11: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import ContextPrecision, ContextRecall, Faithfulness, AnswerCorrectness
/tmp/ipython-input-2621096982.py:11: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import ContextPrecision, Co

In [6]:
datasets = pd.read_excel('/content/drive/MyDrive/RAG/datasets/datasets.xlsx', sheet_name="final")
answers = pd.read_csv('/content/drive/MyDrive/RAG/datasets/adv_rag_with_gemma_result_20260101.csv')

In [7]:
import pandas as pd

# Define a dummy Document class to handle the eval of the string representation
class Document:
    def __init__(self, page_content=None, metadata=None, **kwargs):
        self.page_content = page_content
        self.metadata = metadata
        self.id = kwargs.get('id')

    def __repr__(self):
        return f"Document(page_content='{self.page_content}')"

def parse_sub_questions(row):
    # Initialize default values for up to 3 sub-questions
    result = {}
    for i in range(1, 4):
        result[f'subquestion_{i}'] = None
        result[f'query_{i}'] = None
        result[f'page_content_{i}'] = None

    if pd.isna(row):
        return pd.Series(result)

    try:
        # Evaluate the string to a dictionary, using the dummy Document class
        data = eval(row, {"Document": Document})

        for i in range(1, 4):
            key = f'sub_question_{i}'
            if key in data:
                sub_q = data[key]
                # Mapping extracted fields to requested columns
                # subquestion_i -> mapped to the 'answer' of the sub-question
                # query_i -> mapped to the 'query' of the sub-question
                result[f'subquestion_{i}'] = sub_q.get('answer')
                result[f'query_{i}'] = sub_q.get('query')

                # Extract page content from contexts
                contexts = sub_q.get('contexts', [])
                if isinstance(contexts, list):
                    contents = [doc.page_content for doc in contexts if hasattr(doc, 'page_content')]
                    result[f'page_content_{i}'] = "\n\n".join(contents)
    except Exception as e:
        # In case of parsing errors, return empty results for this row
        pass

    return pd.Series(result)

# Apply the parsing function to the dataframe
expanded_cols = answers['sub_questions'].apply(parse_sub_questions)

# Concatenate with original relevant columns
# We include 'index' by resetting the index, and keep 'query' and 'answer'
answers_df = pd.concat([answers[['query', 'answer']], expanded_cols], axis=1)
answers_df = answers_df.reset_index()

# Select and reorder columns as requested
cols = ['index', 'query', 'answer']
for i in range(1, 4):
    cols.extend([f'subquestion_{i}', f'query_{i}', f'page_content_{i}'])

# Create the final dataframe
answers_df = answers_df[cols]

display(answers_df.head())

,index,query,answer,subquestion_1,query_1,page_content_1,subquestion_2,query_2,page_content_2,subquestion_3,query_3,page_content_3
0,0,Which month is set aside each year in the UAE ...,The UAE designates March annually as the month...,"According to the document, the UAE designates ...",What month does the UAE designate for nationwi...,international reading performance benchmark fo...,"According to the documents, the Abu Dhabi Read...",What is the Abu Dhabi Reads campaign?,international reading performance benchmark fo...,"Yes, the month of March is “dedicated annually...",Are the reading and literacy activities associ...,international reading performance benchmark fo...
1,1,"How many public, private, and charter schools ...","Based on the provided information, there are a...","According to the documents, there are 218 publ...",How many public schools are located in Abu Dhabi?,There are 218 public schools with a student po...,There are 203 private schools located in Abu D...,How many private schools are located in Abu Dh...,There are 218 public schools with a student po...,"According to the documents, there are 22 chart...",How many charter schools are located in Abu Dh...,There are 218 public schools with a student po...
2,2,At which year levels are Australian students a...,NaN,None,None,None,None,None,None,None,None,None
3,3,What grading scale must Australian schools use...,Australian schools utilize a five-point gradin...,"According to the provided document, Australian...",What is the standard grading scale used by Aus...,AUSTRALIA \nPIRLS 2021 ENCYCLOPEDIA 9 \n \n \n...,According to the provided text from the PIRLS ...,What specific academic level or performance do...,AUSTRALIA \nPIRLS 2021 ENCYCLOPEDIA 9 \n \n \n...,None,None,None
4,4,What mandatory early education requirement mus...,Austrian children who fall before March 1st of...,"According to the documents, six-year-old child...",What is the specific type of early education r...,of children whose sixth birthday falls before ...,The document states that six-year-old children...,What is the duration or length of this require...,of children whose sixth birthday falls before ...,"According to the documents, Austrian children ...",What are the criteria or standards that Austri...,of children whose sixth birthday falls before ...


In [8]:
# Create qa_data by combining datasets and answers_df
# Ensure alignment by index as they correspond to the same dataset
qa_data = pd.DataFrame()

# Map columns as requested
qa_data['user_input'] = datasets['question']
qa_data['answer_ref'] = datasets['answer_ref']
qa_data['answer'] = answers_df['answer']
qa_data['reference'] = datasets['passage_1_text']
qa_data['retrieved_contexts_1'] = answers_df['page_content_1']
qa_data['retrieved_contexts_2'] = answers_df['page_content_2']
qa_data['retrieved_contexts_3'] = answers_df['page_content_3']

# Create 'retrieved_contexts' list column for Ragas evaluation
# Filter out None or NaN values
def create_context_list(row):
    contexts = [row['retrieved_contexts_1'], row['retrieved_contexts_2'], row['retrieved_contexts_3']]
    return [c for c in contexts if pd.notna(c) and c is not None]

qa_data['retrieved_contexts'] = qa_data.apply(create_context_list, axis=1)

# Reset index to get the 'index' column
qa_data = qa_data.reset_index()

# Reorder columns to match request (plus retrieved_contexts for compatibility)
desired_columns = ['index', 'user_input', 'answer_ref', 'answer', 'reference',
                   'retrieved_contexts_1', 'retrieved_contexts_2', 'retrieved_contexts_3', 'retrieved_contexts']
qa_data = qa_data[desired_columns]

display(qa_data.head())

,index,user_input,answer_ref,answer,reference,retrieved_contexts_1,retrieved_contexts_2,retrieved_contexts_3,retrieved_contexts
0,0,Which month is set aside each year in the UAE ...,March,The UAE designates March annually as the month...,Building on the Abu Dhabi Reads campaign launc...,international reading performance benchmark fo...,international reading performance benchmark fo...,international reading performance benchmark fo...,[international reading performance benchmark f...
1,1,"How many public, private, and charter schools ...","Abu Dhabi has 218 public schools, 203 private ...","Based on the provided information, there are a...",There are 218 public schools with a student po...,There are 218 public schools with a student po...,There are 218 public schools with a student po...,There are 218 public schools with a student po...,[There are 218 public schools with a student p...
2,2,At which year levels are Australian students a...,Australian students are assessed in NAPLAN in ...,NaN,NAPLAN provides data on the achievement of all...,None,None,None,[]
3,3,What grading scale must Australian schools use...,Australian schools must use a five-point A–E g...,Australian schools utilize a five-point gradin...,All schools are required to report student ach...,AUSTRALIA \nPIRLS 2021 ENCYCLOPEDIA 9 \n \n \n...,AUSTRALIA \nPIRLS 2021 ENCYCLOPEDIA 9 \n \n \n...,None,[AUSTRALIA \nPIRLS 2021 ENCYCLOPEDIA 9 \n \n \...
4,4,What mandatory early education requirement mus...,Austrian children must complete one compulsory...,Austrian children who fall before March 1st of...,"Since 2010, one year of Kindergarten has been ...",of children whose sixth birthday falls before ...,of children whose sixth birthday falls before ...,of children whose sixth birthday falls before ...,[of children whose sixth birthday falls before...


## Evaluation

### Context Precision and Recall

In [9]:
%%time
import asyncio
import nest_asyncio
from ragas import evaluate
from datasets import Dataset
from ragas.run_config import RunConfig

# Ensure data types are consistent for Arrow conversion
# answer_ref caused an ArrowTypeError because it contained integers
qa_data['answer_ref'] = qa_data['answer_ref'].astype(str)
qa_data['user_input'] = qa_data['user_input'].astype(str)
qa_data['answer'] = qa_data['answer'].astype(str)
qa_data['reference'] = qa_data['reference'].astype(str)

# Prepare dataset for Ragas evaluation
# Ragas v0.4+ expects 'response' instead of 'answer'
eval_dataset = Dataset.from_pandas(qa_data.rename(columns={'answer': 'response'}))

# Re-initialize metrics with the compatible classes imported in the setup cell
# We pass the llm wrapper we created earlier
context_precision_scorer = ContextPrecision(llm=llm)
context_recall_scorer = ContextRecall(llm=llm)

nest_asyncio.apply()

# Configure run to be sequential to avoid overwhelming the local Ollama server
# Increase timeout to handle slower local inference (CPU mode)
run_config = RunConfig(max_workers=1, timeout=600)

# Run evaluation using the standard evaluate function
# raise_exceptions=False allows the process to continue even if some rows fail
results = evaluate(
    dataset=eval_dataset,
    metrics=[context_precision_scorer, context_recall_scorer],
    raise_exceptions=False,
    run_config=run_config
)

# Assign scores back to dataframe
qa_data['context_precision_score'] = results['context_precision']
qa_data['context_recall_score'] = results['context_recall']

# Display the first few rows to verify
display(qa_data[['context_precision_score', 'context_recall_score']].head())

Evaluating:   0%|          | 0/600 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[0]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[37]: TimeoutError()
ERROR:ragas.prompt.pydantic_prompt:Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.prompt.pydantic_prompt:Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.prompt.pydantic_prompt:Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.prompt.pydantic_prompt:Prompt context_recall_classification_prompt failed to parse output: The output parser failed to parse the output including retries.
ERROR:ragas.executor:Exception raised in Job[63]: RagasOutputParserException(The output parser failed to parse the output including retries.)
ERROR:ragas.executor:Exception raised in Job[73]: TimeoutError()
ERROR:ragas.executor:Exception raised in Job[81]: Tim

,context_precision_score,context_recall_score
0,NaN,1.0
1,1.0,1.0
2,0.0,0.0
3,1.0,1.0
4,0.0,0.0


CPU times: user 8min 42s, sys: 1min 9s, total: 9min 52s
Wall time: 6h 39min 42s


In [10]:
# Save result to CSV
qa_data.to_csv('/content/drive/MyDrive/RAG/results/eval_adv_precision_recall_gemma_20260105.csv', index=False)

### Faithfulness

In [10]:
%%time
import asyncio
import nest_asyncio
from pydantic import ValidationError

faithfulness_scorer = Faithfulness(llm=llm)
faithfulness_scores = []

async def calculate_faithfulness():
    for index, row in qa_data.iterrows():
        print("Evaluating index: ", index)
        try:
            faithfulness_result = await faithfulness_scorer.ascore(
                user_input=row['user_input'],
                response=row['answer'],
                retrieved_contexts=row['retrieved_contexts'],
            )
            print(faithfulness_result.value)
            faithfulness_scores.append(faithfulness_result.value)
        except (ValueError, ValidationError) as e:
            print(f"Skipping evaluation for index {index} due to: {type(e).__name__} - {e}")
            faithfulness_scores.append(None)
        except Exception as e:
            print(f"An unexpected error occurred at index {index}: {type(e).__name__} - {e}")
            faithfulness_scores.append(None)

nest_asyncio.apply()
asyncio.run(calculate_faithfulness())

qa_data['faithfulness_score'] = faithfulness_scores

Evaluating index:  0
0.4
Evaluating index:  1
0.6666666666666666
Evaluating index:  2
Skipping evaluation for index 2 due to: ValueError - retrieved_contexts is missing. Please add retrieved_contexts to the test sample.
Evaluating index:  3
0.4444444444444444
Evaluating index:  4
0.8
Evaluating index:  5
0.5
Evaluating index:  6
0.9230769230769231
Evaluating index:  7
Skipping evaluation for index 7 due to: ValueError - retrieved_contexts is missing. Please add retrieved_contexts to the test sample.
Evaluating index:  8
0.4
Evaluating index:  9
0.8888888888888888
Evaluating index:  10
0.7
Evaluating index:  11
0.875
Evaluating index:  12
Skipping evaluation for index 12 due to: ValueError - retrieved_contexts is missing. Please add retrieved_contexts to the test sample.
Evaluating index:  13
0.9166666666666666
Evaluating index:  14
0.8571428571428571
Evaluating index:  15
0.07142857142857142
Evaluating index:  16
0.5555555555555556
Evaluating index:  17
Skipping evaluation for index 17

In [11]:
qa_data.columns

Index(['index', 'user_input', 'answer_ref', 'answer', 'reference',
       'retrieved_contexts_1', 'retrieved_contexts_2', 'retrieved_contexts_3',
       'retrieved_contexts', 'faithfulness_score'],
      dtype='object')

In [12]:
# Save result to CSV
qa_data.to_csv('/content/drive/MyDrive/RAG/results/eval_with_adv_faithfulness_gemma_20260102.csv.csv', index=False)


### Answer Correctness

In [13]:
%%time
import asyncio
import nest_asyncio

answer_correctness_scorer = AnswerCorrectness(llm=llm, embeddings=embeddings)
answer_correctness_scores = []

async def calculate_answer_correctness():

    for index, row in qa_data.iterrows():
        print("Evaluating index: ", index)
        answer_correctness_result = await answer_correctness_scorer.ascore(
            user_input=row['user_input'],
            response=row['answer'],
            reference=row['reference'],
        )

        print(answer_correctness_result.value)

        answer_correctness_scores.append(answer_correctness_result.value)

nest_asyncio.apply()
asyncio.run(calculate_answer_correctness())

qa_data['answer_correctness_score'] = answer_correctness_scores

Evaluating index:  0


InstructorRetryException: <failed_attempts>

<generation number="1">
<exception>
    Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
</exception>
<completion>
    None
</completion>
</generation>

<generation number="2">
<exception>
    Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
</exception>
<completion>
    None
</completion>
</generation>

<generation number="3">
<exception>
    Error code: 429 - {'error': {'message': 'You exceeded your current quota, please chec

In [ ]:
# Save result to CSV
qa_data.to_csv('/content/drive/MyDrive/RAG/results/eval_adv_with_answer_correctness_gemma_20260105.csv', index=False)

### Factual Correctness

In [ ]:
%%time
import asyncio
import nest_asyncio

factual_correctness_scorer = FactualCorrectness(llm=llm)
factual_correctness_scores = []

async def calculate_factual_correctness():
    for index, row in qa_data.iterrows():
        factual_correctness_result = await factual_correctness_scorer.ascore(
            response=row['answer'],
            reference=row['reference'],
            )

        factual_correctness_scores.append(factual_correctness_result.value)

nest_asyncio.apply()
asyncio.run(calculate_factual_correctness())

qa_data['factual_correctness_score'] = factual_correctness_scores

ValidationError: 1 validation error for ClaimDecompositionInput
response
  Input should be a valid string [type=string_type, input_value=nan, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

In [ ]:
# Save result to CSV
qa_data.to_csv('/content/drive/MyDrive/RAG/results/eval_adv_with_factual_correctness_gemma_20260105.csv', index=False)